# NB5 · Forecast 2024 y escenarios

## Objetivo

Proyectar las principales magnitudes financieras de 2024 con modelos simples y defendibles.

Este notebook:

- compara dos métodos de forecast;
- selecciona el modelo con menor error histórico;
- proyecta 12 meses;
- construye escenarios base, optimista y pesimista;
- recalcula resultados y ratios previstos;
- deja las tablas preparadas para Power BI.

> Con 36 meses de datos, se priorizan modelos sencillos frente a modelos complejos con riesgo de sobreajuste.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

from IPython.display import display


pd.set_option("display.max_columns", 120)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Debe existir una carpeta 'data/' en la raíz."
    )


PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "05"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUTAS = {
    "pyg": (
        PROCESSED_DIR
        / "03"
        / "03_magnitudes_pyg.csv"
    ),

    "base_financiera": (
        PROCESSED_DIR
        / "04"
        / "04_base_financiera.csv"
    ),
}


# ==========================================================
# COMPROBACIÓN
# ==========================================================

for nombre, ruta in RUTAS.items():

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró '{ruta.name}'. "
            "Ejecuta primero NB3 y NB4."
        )


print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Salida NB5:         {OUTPUT_DIR}")

print("\nArchivos encontrados:")

for nombre, ruta in RUTAS.items():
    print(f" - {nombre}: {ruta.name}")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Salida NB5:         C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\05

Archivos encontrados:
 - pyg: 03_magnitudes_pyg.csv
 - base_financiera: 04_base_financiera.csv


In [2]:
# ==========================================================
# 2. CARGA Y BASE HISTÓRICA
# ==========================================================

pyg = pd.read_csv(
    RUTAS["pyg"],
    parse_dates=["periodo"]
)

base_financiera = pd.read_csv(
    RUTAS["base_financiera"],
    parse_dates=["periodo"]
)


# ==========================================================
# VARIABLES PYG
# ==========================================================

columnas_pyg = [
    "periodo",
    "ingresos_explotacion",
    "aprovisionamientos",
    "gastos_personal",
    "otros_gastos_operativos",
    "amortizaciones",
    "deterioros",
    "resultado_financiero",
    "resultado_no_recurrente",
    "impuesto_beneficios",
]


# ==========================================================
# VARIABLES BALANCE
# ==========================================================

columnas_balance = [
    "periodo",
    "activo_corriente",
    "activo_no_corriente",
    "pasivo_corriente",
    "pasivo_no_corriente",
    "tesoreria",
    "clientes",
    "proveedores",
    "existencias",
    "deuda_financiera_bruta",
]


# ==========================================================
# VALIDACIÓN DE COLUMNAS
# ==========================================================

for columna in columnas_pyg:

    if columna not in pyg.columns:
        raise ValueError(
            f"Falta la columna '{columna}' "
            "en 03_magnitudes_pyg.csv."
        )


for columna in columnas_balance:

    if columna not in base_financiera.columns:
        raise ValueError(
            f"Falta la columna '{columna}' "
            "en 04_base_financiera.csv."
        )


# ==========================================================
# BASE HISTÓRICA
# ==========================================================

historico = (
    pyg[columnas_pyg]
    .merge(
        base_financiera[columnas_balance],
        on="periodo",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("periodo")
    .reset_index(drop=True)
)


# ==========================================================
# VALIDACIONES
# ==========================================================

if historico["periodo"].duplicated().any():
    raise ValueError(
        "Existen meses duplicados."
    )


periodos_esperados = pd.date_range(
    historico["periodo"].min(),
    historico["periodo"].max(),
    freq="MS"
)


periodos_faltantes = (
    periodos_esperados
    .difference(
        pd.DatetimeIndex(
            historico["periodo"]
        )
    )
)


if len(periodos_faltantes) > 0:
    raise ValueError(
        "Faltan meses en el histórico: "
        f"{[
            x.strftime('%Y-%m')
            for x in periodos_faltantes
        ]}"
    )


# Necesitamos suficiente historia para entrenar y reservar 12 meses para validación.

MIN_MESES = 24

if len(historico) < MIN_MESES:
    raise ValueError(
        f"Se necesitan al menos {MIN_MESES} meses "
        "para generar el forecast."
    )


print(
    f"Periodo histórico: "
    f"{historico['periodo'].min():%Y-%m} "
    f"→ "
    f"{historico['periodo'].max():%Y-%m}"
)

print(
    f"Meses disponibles: {len(historico)}"
)

display(historico.tail())

Periodo histórico: 2021-01 → 2023-12
Meses disponibles: 36


,periodo,ingresos_explotacion,aprovisionamientos,gastos_personal,otros_gastos_operativos,amortizaciones,deterioros,resultado_financiero,resultado_no_recurrente,impuesto_beneficios,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,tesoreria,clientes,proveedores,existencias,deuda_financiera_bruta
31,2023-08-01,"1,085,682.81","-26,341.28","-271,522.97","-592,456.17","-150,637.40",0.00,0.00,0.00,"-11,181.25","6,156,899.87","18,924,019.36","2,887,783.97","18,252,366.73","5,715,460.23","36,335.84","1,905,031.30",0.00,"18,278,217.06"
32,2023-09-01,"1,224,763.29","-29,946.08","-271,917.64","-559,067.46","-151,542.27",0.00,"-218,508.70","-10,938.84","4,289.42","5,675,796.76","18,849,240.53","2,344,770.31","18,252,366.73","4,936,971.05","63,020.96","1,437,569.29",0.00,"18,280,371.70"
33,2023-10-01,"1,149,896.72","-22,914.71","-281,203.93","-568,543.32","-152,169.18",0.00,"-1,750.06","-5,400.60","-29,478.73","5,471,808.90","18,734,933.86","1,938,039.59","18,252,366.73","5,265,577.98","52,795.10","1,028,342.43",0.00,"18,280,371.70"
34,2023-11-01,"1,194,151.60","-24,999.52","-273,188.02","-565,018.21","-145,920.10",0.00,-85.37,-539.69,"-46,100.17","5,932,927.02","18,612,690.51","2,138,613.84","18,252,366.73","5,703,230.63","56,001.41","1,135,291.75",0.00,"18,278,213.06"
35,2023-12-01,"979,435.02","-47,911.85","-256,598.35","-313,553.89","-146,633.73",0.00,"-213,522.12","-4,882.83","24,150.81","4,718,524.50","18,514,451.31","2,057,855.79","17,000,000.00","4,471,153.83","87,038.57","1,079,476.28",0.00,"17,017,811.77"


In [3]:
# ==========================================================
# 3. MODELOS ML SIMPLES
# ==========================================================

def crear_features_tendencia(periodos):
    """
    Crea únicamente una variable temporal.
    """

    n = len(periodos)

    return pd.DataFrame({
        "tendencia": np.arange(n)
    })


def crear_features_estacionales(periodos):
    """
    Crea tendencia + mes del año.
    """

    n = len(periodos)

    datos = pd.DataFrame({
        "tendencia": np.arange(n),
        "mes": periodos.dt.month
    })

    return pd.get_dummies(
        datos,
        columns=["mes"],
        drop_first=False,
        dtype=float
    )


def forecast_regresion(
    serie,
    periodos,
    horizonte=12,
    estacional=False
):
    """
    Regresión lineal simple.
    """

    if estacional:

        X = crear_features_estacionales(
            periodos
        )

    else:

        X = crear_features_tendencia(
            periodos
        )


    modelo = LinearRegression()

    modelo.fit(
        X,
        serie.astype(float)
    )


    fecha_inicio = (
        periodos.max()
        + pd.offsets.MonthBegin(1)
    )

    periodos_futuros = pd.Series(
        pd.date_range(
            fecha_inicio,
            periods=horizonte,
            freq="MS"
        )
    )


    periodos_completos = pd.concat(
        [
            periodos.reset_index(drop=True),
            periodos_futuros
        ],
        ignore_index=True
    )


    if estacional:

        X_completo = crear_features_estacionales(
            periodos_completos
        )

    else:

        X_completo = crear_features_tendencia(
            periodos_completos
        )


    X_futuro = (
        X_completo
        .iloc[-horizonte:]
        .copy()
    )

    # Se garantizan las mismas columnas que en el entrenamiento.
    X_futuro = X_futuro.reindex(
        columns=X.columns,
        fill_value=0
    )

    return modelo.predict(
        X_futuro
    )


MODELOS = {
    "Tendencia": False,
    "Tendencia + estacionalidad": True,
}

In [4]:
# ==========================================================
# 4. BACKTESTING Y SELECCIÓN DEL MODELO
# ==========================================================

HORIZONTE_VALIDACION = 12


def evaluar_modelos(
    serie,
    periodos
):

    entrenamiento = (
        serie.iloc[
            :-HORIZONTE_VALIDACION
        ]
        .reset_index(drop=True)
    )

    periodos_entrenamiento = (
        periodos.iloc[
            :-HORIZONTE_VALIDACION
        ]
        .reset_index(drop=True)
    )

    validacion = (
        serie.iloc[
            -HORIZONTE_VALIDACION:
        ]
        .reset_index(drop=True)
    )

    real = validacion.to_numpy(
        dtype=float
    )


    resultados = []


    for nombre, estacional in MODELOS.items():

        prediccion = forecast_regresion(
            entrenamiento,
            periodos_entrenamiento,
            horizonte=HORIZONTE_VALIDACION,
            estacional=estacional
        )


        mae = mean_absolute_error(
            real,
            prediccion
        )


        resultados.append({
            "modelo": nombre,
            "mae": mae,
        })


    return (
        pd.DataFrame(resultados)
        .sort_values("mae")
        .reset_index(drop=True)
    )


# ==========================================================
# VARIABLES
# ==========================================================

variables_modelo = [
    columna
    for columna in historico.columns
    if columna != "periodo"
]


resultados_modelos = []
modelos_elegidos = {}


for variable in variables_modelo:

    serie = (
        historico[variable]
        .astype(float)
        .fillna(0.0)
    )


    evaluacion = evaluar_modelos(
        serie,
        historico["periodo"]
    )


    mejor_modelo = (
        evaluacion.iloc[0][
            "modelo"
        ]
    )


    modelos_elegidos[
        variable
    ] = mejor_modelo


    evaluacion[
        "variable"
    ] = variable


    evaluacion[
        "modelo_elegido"
    ] = (
        evaluacion[
            "modelo"
        ]
        .eq(mejor_modelo)
    )


    resultados_modelos.append(
        evaluacion
    )


resultados_modelos = pd.concat(
    resultados_modelos,
    ignore_index=True
)


seleccion_modelos = (
    resultados_modelos.loc[
        resultados_modelos[
            "modelo_elegido"
        ]
    ][[
        "variable",
        "modelo",
        "mae",
    ]]
    .sort_values("variable")
    .reset_index(drop=True)
)

display(seleccion_modelos)

,variable,modelo,mae
0,activo_corriente,Tendencia + estacionalidad,"490,650.42"
1,activo_no_corriente,Tendencia,"703,492.13"
2,amortizaciones,Tendencia,"2,977.25"
3,aprovisionamientos,Tendencia,"3,645.75"
4,clientes,Tendencia,"12,203.50"
5,deterioros,Tendencia,0.00
6,deuda_financiera_bruta,Tendencia,"1,500,462.18"
7,existencias,Tendencia,0.00
8,gastos_personal,Tendencia,"10,274.77"
9,impuesto_beneficios,Tendencia,"44,325.47"


In [5]:
# ==========================================================
# 5. FORECAST BASE
# ==========================================================

HORIZONTE_FORECAST = 12


fecha_inicio = (
    historico[
        "periodo"
    ].max()
    + pd.offsets.MonthBegin(1)
)

periodos_forecast = pd.date_range(
    start=fecha_inicio,
    periods=HORIZONTE_FORECAST,
    freq="MS"
)

forecast_base = pd.DataFrame({
    "periodo":
        periodos_forecast
})


# ==========================================================
# FORECAST DE CADA VARIABLE
# ==========================================================

for variable in variables_modelo:

    modelo_elegido = (
        modelos_elegidos[
            variable
        ]
    )

    estacional = (
        MODELOS[
            modelo_elegido
        ]
    )

    forecast_base[
        variable
    ] = forecast_regresion(
        historico[
            variable
        ].astype(float),
        historico[
            "periodo"
        ],
        horizonte=HORIZONTE_FORECAST,
        estacional=estacional
    )


# ==========================================================
# VARIABLES NO NEGATIVAS
# ==========================================================

columnas_no_negativas = [
    "activo_corriente",
    "activo_no_corriente",
    "pasivo_corriente",
    "pasivo_no_corriente",
    "tesoreria",
    "clientes",
    "proveedores",
    "existencias",
    "deuda_financiera_bruta",
]


for columna in columnas_no_negativas:

    forecast_base[
        columna
    ] = (
        forecast_base[
            columna
        ]
        .clip(lower=0)
    )

forecast_base[
    "escenario"
] = "Base"

print(
    f"Forecast: "
    f"{forecast_base['periodo'].min():%Y-%m} "
    f"→ "
    f"{forecast_base['periodo'].max():%Y-%m}"
)

display(forecast_base.head())

Forecast: 2024-01 → 2024-12


,periodo,ingresos_explotacion,aprovisionamientos,gastos_personal,otros_gastos_operativos,amortizaciones,deterioros,resultado_financiero,resultado_no_recurrente,impuesto_beneficios,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,tesoreria,clientes,proveedores,existencias,deuda_financiera_bruta,escenario
0,2024-01-01,"1,286,061.64","-30,890.73","-282,519.72","-535,223.23","-148,490.74",0.00,"108,561.90","-2,826.29","-46,003.56","6,386,053.41","19,036,531.09","2,620,323.22","18,566,389.47","5,891,498.93","61,533.64","1,392,248.14",0.00,"18,603,530.84",Base
1,2024-02-01,"1,297,026.30","-31,210.31","-284,654.25","-537,280.56","-148,267.98",0.00,"110,967.99","-2,879.78","-47,408.13","6,629,086.18","18,979,279.74","2,679,676.58","18,417,601.46","5,982,237.81","62,027.23","1,392,392.22",0.00,"18,455,837.81",Base
2,2024-03-01,"1,307,990.95","-31,529.88","-286,788.79","-539,337.88","-148,045.22",0.00,"-139,010.89","-2,933.27","-48,812.69","6,656,024.30","18,922,028.38","2,956,487.97","18,268,813.45","6,072,976.70","62,520.81","1,520,892.68",0.00,"18,308,144.78",Base
3,2024-04-01,"1,318,955.60","-31,849.45","-288,923.33","-541,395.21","-147,822.46",0.00,"111,297.58","-2,986.77","-50,217.26","6,561,324.51","18,864,777.03","2,691,909.60","18,120,025.45","6,163,715.58","63,014.39","1,347,551.01",0.00,"18,160,451.76",Base
4,2024-05-01,"1,329,920.25","-32,169.03","-291,057.86","-543,452.53","-147,599.70",0.00,"496,962.84","-3,040.26","-51,621.82","7,656,773.48","18,807,525.67","2,961,116.03","17,971,237.44","6,254,454.46","63,507.98","1,389,317.93",0.00,"18,012,758.73",Base


In [6]:
# ==========================================================
# 6. ESCENARIOS
# ==========================================================

# Los escenarios se aplican sobre el forecast Base.

# Optimista: +5% ingresos, -2% presión de costes, +10% caja, -5% deuda
# Pesimista: -5% ingresos, +3% presión de costes, -10% caja, +8% deuda

def crear_escenario(
    base,
    nombre
):
    escenario = base.copy()

    escenario[
        "escenario"
    ] = nombre


    if nombre == "Optimista":

        escenario[
            "ingresos_explotacion"
        ] *= 1.05

        escenario[
            "aprovisionamientos"
        ] *= 0.98

        escenario[
            "gastos_personal"
        ] *= 0.99

        escenario[
            "otros_gastos_operativos"
        ] *= 0.98


        # Circulante

        tesoreria_base = (
            escenario[
                "tesoreria"
            ].copy()
        )

        clientes_base = (
            escenario[
                "clientes"
            ].copy()
        )

        deuda_base = (
            escenario[
                "deuda_financiera_bruta"
            ].copy()
        )

        escenario[
            "tesoreria"
        ] *= 1.10

        escenario[
            "clientes"
        ] *= 0.95

        escenario[
            "deuda_financiera_bruta"
        ] *= 0.95


        # Se ajusta el activo corriente por los cambios realizados.

        escenario[
            "activo_corriente"
        ] += (
            escenario[
                "tesoreria"
            ]
            - tesoreria_base
        )

        escenario[
            "activo_corriente"
        ] += (
            escenario[
                "clientes"
            ]
            - clientes_base
        )


        # La variación de deuda se refleja en el pasivo no corriente

        escenario[
            "pasivo_no_corriente"
        ] += (
            escenario[
                "deuda_financiera_bruta"
            ]
            - deuda_base
        )


    elif nombre == "Pesimista":

        escenario[
            "ingresos_explotacion"
        ] *= 0.95


        # Mayor presión de costes.

        escenario[
            "aprovisionamientos"
        ] *= 1.03

        escenario[
            "gastos_personal"
        ] *= 1.02

        escenario[
            "otros_gastos_operativos"
        ] *= 1.03


        tesoreria_base = (
            escenario[
                "tesoreria"
            ].copy()
        )

        clientes_base = (
            escenario[
                "clientes"
            ].copy()
        )

        deuda_base = (
            escenario[
                "deuda_financiera_bruta"
            ].copy()
        )

        escenario[
            "tesoreria"
        ] *= 0.90

        escenario[
            "clientes"
        ] *= 1.08

        escenario[
            "deuda_financiera_bruta"
        ] *= 1.08

        escenario[
            "activo_corriente"
        ] += (
            escenario[
                "tesoreria"
            ]
            - tesoreria_base
        )

        escenario[
            "activo_corriente"
        ] += (
            escenario[
                "clientes"
            ]
            - clientes_base
        )


        # La variación de deuda se refleja en el pasivo no corriente

        escenario[
            "pasivo_no_corriente"
        ] += (
            escenario[
                "deuda_financiera_bruta"
            ]
            - deuda_base
        )

    return escenario

forecast_escenarios = pd.concat(
    [
        forecast_base,

        crear_escenario(
            forecast_base,
            "Optimista"
        ),

        crear_escenario(
            forecast_base,
            "Pesimista"
        ),
    ],
    ignore_index=True
)


display(
    forecast_escenarios.head()
)

,periodo,ingresos_explotacion,aprovisionamientos,gastos_personal,otros_gastos_operativos,amortizaciones,deterioros,resultado_financiero,resultado_no_recurrente,impuesto_beneficios,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,tesoreria,clientes,proveedores,existencias,deuda_financiera_bruta,escenario
0,2024-01-01,"1,286,061.64","-30,890.73","-282,519.72","-535,223.23","-148,490.74",0.00,"108,561.90","-2,826.29","-46,003.56","6,386,053.41","19,036,531.09","2,620,323.22","18,566,389.47","5,891,498.93","61,533.64","1,392,248.14",0.00,"18,603,530.84",Base
1,2024-02-01,"1,297,026.30","-31,210.31","-284,654.25","-537,280.56","-148,267.98",0.00,"110,967.99","-2,879.78","-47,408.13","6,629,086.18","18,979,279.74","2,679,676.58","18,417,601.46","5,982,237.81","62,027.23","1,392,392.22",0.00,"18,455,837.81",Base
2,2024-03-01,"1,307,990.95","-31,529.88","-286,788.79","-539,337.88","-148,045.22",0.00,"-139,010.89","-2,933.27","-48,812.69","6,656,024.30","18,922,028.38","2,956,487.97","18,268,813.45","6,072,976.70","62,520.81","1,520,892.68",0.00,"18,308,144.78",Base
3,2024-04-01,"1,318,955.60","-31,849.45","-288,923.33","-541,395.21","-147,822.46",0.00,"111,297.58","-2,986.77","-50,217.26","6,561,324.51","18,864,777.03","2,691,909.60","18,120,025.45","6,163,715.58","63,014.39","1,347,551.01",0.00,"18,160,451.76",Base
4,2024-05-01,"1,329,920.25","-32,169.03","-291,057.86","-543,452.53","-147,599.70",0.00,"496,962.84","-3,040.26","-51,621.82","7,656,773.48","18,807,525.67","2,961,116.03","17,971,237.44","6,254,454.46","63,507.98","1,389,317.93",0.00,"18,012,758.73",Base


In [7]:
# ==========================================================
# 7. ESTADOS FINANCIEROS FORECAST
# ==========================================================

# ==========================================================
# VALIDACIÓN DE COLUMNAS NECESARIAS
# ==========================================================

columnas_necesarias = [
    "ingresos_explotacion",
    "aprovisionamientos",
    "gastos_personal",
    "otros_gastos_operativos",
    "amortizaciones",
    "deterioros",
    "resultado_financiero",
    "resultado_no_recurrente",
    "impuesto_beneficios",
    "activo_corriente",
    "activo_no_corriente",
    "pasivo_corriente",
    "pasivo_no_corriente",
    "tesoreria",
    "deuda_financiera_bruta",
]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in forecast_escenarios.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan columnas necesarias para construir "
        f"los estados forecast: {columnas_faltantes}"
    )


# ==========================================================
# PYG
# ==========================================================

forecast_escenarios["margen_bruto"] = (
    forecast_escenarios["ingresos_explotacion"]
    + forecast_escenarios["aprovisionamientos"]
)


forecast_escenarios["ebitda"] = (
    forecast_escenarios["ingresos_explotacion"]
    + forecast_escenarios["aprovisionamientos"]
    + forecast_escenarios["gastos_personal"]
    + forecast_escenarios["otros_gastos_operativos"]
)


forecast_escenarios["ebit"] = (
    forecast_escenarios["ebitda"]
    + forecast_escenarios["amortizaciones"]
    + forecast_escenarios["deterioros"]
)


forecast_escenarios["resultado_antes_impuestos"] = (
    forecast_escenarios["ebit"]
    + forecast_escenarios["resultado_financiero"]
    + forecast_escenarios["resultado_no_recurrente"]
)


forecast_escenarios["resultado_neto"] = (
    forecast_escenarios["resultado_antes_impuestos"]
    + forecast_escenarios["impuesto_beneficios"]
)


# ==========================================================
# BALANCE
# ==========================================================

forecast_escenarios["activo_total"] = (
    forecast_escenarios["activo_corriente"]
    + forecast_escenarios["activo_no_corriente"]
)


forecast_escenarios["pasivo_total"] = (
    forecast_escenarios["pasivo_corriente"]
    + forecast_escenarios["pasivo_no_corriente"]
)


# Patrimonio neto calculado de forma residual para que cuadre el balance (en otros NBs se ha calculado de forma estricta, aquí no es necesaria tanta exigencia)

forecast_escenarios["patrimonio_neto"] = (
    forecast_escenarios["activo_total"]
    - forecast_escenarios["pasivo_total"]
)


forecast_escenarios["deuda_financiera_neta"] = (
    forecast_escenarios["deuda_financiera_bruta"]
    - forecast_escenarios["tesoreria"]
)


forecast_escenarios["fondo_maniobra"] = (
    forecast_escenarios["activo_corriente"]
    - forecast_escenarios["pasivo_corriente"]
)


# ==========================================================
# CONTROL DE CUADRE
# ==========================================================

forecast_escenarios["diferencia_balance"] = (
    forecast_escenarios["activo_total"]
    - (
        forecast_escenarios["pasivo_total"]
        + forecast_escenarios["patrimonio_neto"]
    )
)


forecast_escenarios.loc[
    forecast_escenarios["diferencia_balance"].abs() <= 0.01,
    "diferencia_balance"
] = 0.0


if (
    forecast_escenarios["diferencia_balance"]
    .abs()
    .max()
    > 0.01
):
    raise ValueError(
        "El Balance forecast no cuadra."
    )


# ==========================================================
# CONTROL FINAL
# ==========================================================

n_controles = len(forecast_escenarios)

print(
    f"Control forecast: "
    f"{n_controles}/{n_controles} escenarios-mes cuadran."
)

print(
    "Magnitudes calculadas: "
    "Margen bruto, EBITDA, EBIT, Resultado neto, "
    "Activo, Pasivo y Patrimonio Neto."
)

display(
    forecast_escenarios[[
        "periodo",
        "escenario",
        "ingresos_explotacion",
        "margen_bruto",
        "ebitda",
        "ebit",
        "resultado_neto",
        "activo_total",
        "pasivo_total",
        "patrimonio_neto",
    ]].head()
)

Control forecast: 36/36 escenarios-mes cuadran.
Magnitudes calculadas: Margen bruto, EBITDA, EBIT, Resultado neto, Activo, Pasivo y Patrimonio Neto.


,periodo,escenario,ingresos_explotacion,margen_bruto,ebitda,ebit,resultado_neto,activo_total,pasivo_total,patrimonio_neto
0,2024-01-01,Base,"1,286,061.64","1,255,170.91","437,427.96","288,937.22","348,669.27","25,422,584.50","21,186,712.69","4,235,871.81"
1,2024-02-01,Base,"1,297,026.30","1,265,815.99","443,881.18","295,613.20","356,293.28","25,608,365.91","21,097,278.05","4,511,087.86"
2,2024-03-01,Base,"1,307,990.95","1,276,461.07","450,334.39","302,289.17","111,532.32","25,578,052.68","21,225,301.42","4,352,751.26"
3,2024-04-01,Base,"1,318,955.60","1,287,106.15","456,787.61","308,965.15","367,058.70","25,426,101.53","20,811,935.04","4,614,166.49"
4,2024-05-01,Base,"1,329,920.25","1,297,751.22","463,240.83","315,641.13","757,941.88","26,464,299.15","20,932,353.46","5,531,945.68"


In [8]:
# ==========================================================
# 8. RATIOS FORECAST
# ==========================================================

TOLERANCIA = 0.000001

def dividir(
    numerador,
    denominador
):
    """
    División segura.
    """

    return np.where(
        denominador.abs() > TOLERANCIA,
        numerador / denominador,
        np.nan
    )


# ==========================================================
# BASE
# ==========================================================

ratios_forecast = (
    forecast_escenarios[[
        "periodo",
        "escenario",
    ]]
    .copy()
)


# ==========================================================
# RENTABILIDAD
# ==========================================================

ratios_forecast[
    "margen_bruto"
] = dividir(
    forecast_escenarios[
        "margen_bruto"
    ],
    forecast_escenarios[
        "ingresos_explotacion"
    ]
)

ratios_forecast[
    "margen_ebitda"
] = dividir(
    forecast_escenarios[
        "ebitda"
    ],
    forecast_escenarios[
        "ingresos_explotacion"
    ]
)

ratios_forecast[
    "margen_ebit"
] = dividir(
    forecast_escenarios[
        "ebit"
    ],
    forecast_escenarios[
        "ingresos_explotacion"
    ]
)

ratios_forecast[
    "margen_neto"
] = dividir(
    forecast_escenarios[
        "resultado_neto"
    ],
    forecast_escenarios[
        "ingresos_explotacion"
    ]
)


# ==========================================================
# LIQUIDEZ
# ==========================================================

ratios_forecast[
    "liquidez_corriente"
] = dividir(
    forecast_escenarios[
        "activo_corriente"
    ],
    forecast_escenarios[
        "pasivo_corriente"
    ]
)

ratios_forecast[
    "prueba_acida"
] = dividir(
    forecast_escenarios[
        "activo_corriente"
    ]
    - forecast_escenarios[
        "existencias"
    ],
    forecast_escenarios[
        "pasivo_corriente"
    ]
)

ratios_forecast[
    "liquidez_inmediata"
] = dividir(
    forecast_escenarios[
        "tesoreria"
    ],
    forecast_escenarios[
        "pasivo_corriente"
    ]
)


# ==========================================================
# SOLVENCIA
# ==========================================================

ratios_forecast[
    "endeudamiento"
] = dividir(
    forecast_escenarios[
        "pasivo_total"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)

ratios_forecast[
    "autonomia_financiera"
] = dividir(
    forecast_escenarios[
        "patrimonio_neto"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)

ratios_forecast[
    "deuda_sobre_patrimonio"
] = np.where(
    forecast_escenarios[
        "patrimonio_neto"
    ] > TOLERANCIA,

    forecast_escenarios[
        "deuda_financiera_bruta"
    ]
    / forecast_escenarios[
        "patrimonio_neto"
    ],

    np.nan
)

ratios_forecast[
    "deuda_financiera_sobre_activo"
] = dividir(
    forecast_escenarios[
        "deuda_financiera_bruta"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)

ratios_forecast[
    "tesoreria_sobre_activo"
] = dividir(
    forecast_escenarios[
        "tesoreria"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)

ratios_forecast[
    "fondo_maniobra_sobre_activo"
] = dividir(
    forecast_escenarios[
        "fondo_maniobra"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)

ratios_forecast[
    "existencias_sobre_activo"
] = dividir(
    forecast_escenarios[
        "existencias"
    ],
    forecast_escenarios[
        "activo_total"
    ]
)


# ==========================================================
# FECHAS
# ==========================================================

ratios_forecast["año"] = (
    ratios_forecast[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

ratios_forecast[
    "mes_numero"
] = (
    ratios_forecast[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

ratios_forecast[
    "año_mes"
] = (
    ratios_forecast[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# LIMPIEZA
# ==========================================================

ratios_forecast.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
    inplace=True
)

display(
    ratios_forecast.head()
)

,periodo,escenario,margen_bruto,margen_ebitda,margen_ebit,margen_neto,liquidez_corriente,prueba_acida,liquidez_inmediata,endeudamiento,autonomia_financiera,deuda_sobre_patrimonio,deuda_financiera_sobre_activo,tesoreria_sobre_activo,fondo_maniobra_sobre_activo,existencias_sobre_activo,año,mes_numero,año_mes
0,2024-01-01,Base,0.98,0.34,0.22,0.27,2.44,2.44,2.25,0.83,0.17,4.39,0.73,0.23,0.15,0.00,2024,1,2024-01
1,2024-02-01,Base,0.98,0.34,0.23,0.27,2.47,2.47,2.23,0.82,0.18,4.09,0.72,0.23,0.15,0.00,2024,2,2024-02
2,2024-03-01,Base,0.98,0.34,0.23,0.09,2.25,2.25,2.05,0.83,0.17,4.21,0.72,0.24,0.14,0.00,2024,3,2024-03
3,2024-04-01,Base,0.98,0.35,0.23,0.28,2.44,2.44,2.29,0.82,0.18,3.94,0.71,0.24,0.15,0.00,2024,4,2024-04
4,2024-05-01,Base,0.98,0.35,0.24,0.57,2.59,2.59,2.11,0.79,0.21,3.26,0.68,0.24,0.18,0.00,2024,5,2024-05


In [9]:
# ==========================================================
# 9. TABLAS PARA POWER BI
# ==========================================================

# ==========================================================
# VARIABLES TEMPORALES FORECAST
# ==========================================================

forecast_escenarios[
    "año"
] = (
    forecast_escenarios[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

forecast_escenarios[
    "mes_numero"
] = (
    forecast_escenarios[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

forecast_escenarios[
    "año_mes"
] = (
    forecast_escenarios[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


# ==========================================================
# MÉTRICAS FORECAST
# ==========================================================

metricas_forecast = [
    "ingresos_explotacion",
    "margen_bruto",
    "ebitda",
    "ebit",
    "resultado_neto",
    "activo_total",
    "pasivo_total",
    "patrimonio_neto",
    "tesoreria",
    "deuda_financiera_bruta",
    "deuda_financiera_neta",
    "clientes",
    "proveedores",
    "existencias",
    "fondo_maniobra",
]


forecast_powerbi = (
    forecast_escenarios
    .melt(
        id_vars=[
            "periodo",
            "año",
            "mes_numero",
            "año_mes",
            "escenario",
        ],
        value_vars=metricas_forecast,
        var_name="metrica",
        value_name="valor"
    )
)


# ==========================================================
# RATIOS FORECAST
# ==========================================================

columnas_ratios = [
    columna
    for columna in ratios_forecast.columns
    if columna not in [
        "periodo",
        "año",
        "mes_numero",
        "año_mes",
        "escenario",
    ]
]

ratios_powerbi = (
    ratios_forecast
    .melt(
        id_vars=[
            "periodo",
            "año",
            "mes_numero",
            "año_mes",
            "escenario",
        ],
        value_vars=columnas_ratios,
        var_name="ratio",
        value_name="valor"
    )
)


# ==========================================================
# CONTROL
# ==========================================================

if forecast_powerbi.empty:
    raise ValueError(
        "La tabla forecast_powerbi está vacía."
    )

if ratios_powerbi.empty:
    raise ValueError(
        "La tabla ratios_powerbi está vacía."
    )


print(
    f"Filas forecast Power BI: "
    f"{len(forecast_powerbi):,}"
)

print(
    f"Filas ratios Power BI: "
    f"{len(ratios_powerbi):,}"
)

display(
    forecast_powerbi.head()
)

display(
    ratios_powerbi.head()
)

Filas forecast Power BI: 540
Filas ratios Power BI: 504


,periodo,año,mes_numero,año_mes,escenario,metrica,valor
0,2024-01-01,2024,1,2024-01,Base,ingresos_explotacion,"1,286,061.64"
1,2024-02-01,2024,2,2024-02,Base,ingresos_explotacion,"1,297,026.30"
2,2024-03-01,2024,3,2024-03,Base,ingresos_explotacion,"1,307,990.95"
3,2024-04-01,2024,4,2024-04,Base,ingresos_explotacion,"1,318,955.60"
4,2024-05-01,2024,5,2024-05,Base,ingresos_explotacion,"1,329,920.25"


,periodo,año,mes_numero,año_mes,escenario,ratio,valor
0,2024-01-01,2024,1,2024-01,Base,margen_bruto,0.98
1,2024-02-01,2024,2,2024-02,Base,margen_bruto,0.98
2,2024-03-01,2024,3,2024-03,Base,margen_bruto,0.98
3,2024-04-01,2024,4,2024-04,Base,margen_bruto,0.98
4,2024-05-01,2024,5,2024-05,Base,margen_bruto,0.98


In [10]:
# ==========================================================
# 10. RESUMEN DE ESCENARIOS
# ==========================================================

resumen_escenarios = (
    forecast_escenarios
    .sort_values(
        "periodo"
    )
    .groupby(
        "escenario",
        as_index=False,
        observed=True
    )
    .agg(
        ingresos_forecast=(
            "ingresos_explotacion",
            "sum"
        ),

        ebitda_forecast=(
            "ebitda",
            "sum"
        ),

        resultado_neto_forecast=(
            "resultado_neto",
            "sum"
        ),

        activo_final=(
            "activo_total",
            "last"
        ),

        pasivo_final=(
            "pasivo_total",
            "last"
        ),

        patrimonio_final=(
            "patrimonio_neto",
            "last"
        ),

        caja_final=(
            "tesoreria",
            "last"
        ),

        deuda_final=(
            "deuda_financiera_bruta",
            "last"
        ),

        clientes_final=(
            "clientes",
            "last"
        ),

        proveedores_final=(
            "proveedores",
            "last"
        ),

        fondo_maniobra_final=(
            "fondo_maniobra",
            "last"
        ),
    )
)


# ==========================================================
# MÁRGENES DEL PERIODO FORECAST
# ==========================================================

resumen_escenarios[
    "margen_ebitda_forecast"
] = dividir(
    resumen_escenarios[
        "ebitda_forecast"
    ],
    resumen_escenarios[
        "ingresos_forecast"
    ]
)


resumen_escenarios[
    "margen_neto_forecast"
] = dividir(
    resumen_escenarios[
        "resultado_neto_forecast"
    ],
    resumen_escenarios[
        "ingresos_forecast"
    ]
)


# ==========================================================
# ORDEN DE ESCENARIOS
# ==========================================================

orden_escenarios = pd.CategoricalDtype(
    categories=[
        "Pesimista",
        "Base",
        "Optimista",
    ],
    ordered=True
)

resumen_escenarios[
    "escenario"
] = (
    resumen_escenarios[
        "escenario"
    ]
    .astype(
        orden_escenarios
    )
)

resumen_escenarios = (
    resumen_escenarios
    .sort_values(
        "escenario"
    )
    .reset_index(drop=True)
)

display(
    resumen_escenarios
)

,escenario,ingresos_forecast,ebitda_forecast,resultado_neto_forecast,activo_final,pasivo_final,patrimonio_final,caja_final,deuda_final,clientes_final,proveedores_final,fondo_maniobra_final,margen_ebitda_forecast,margen_neto_forecast
0,Pesimista,"15,348,586.39","4,588,097.86","2,838,612.48","24,414,761.62","21,136,579.05","3,278,182.57","6,200,663.98","18,337,220.14","72,320.10","1,420,978.58","3,159,450.36",0.30,0.18
1,Base,"16,156,406.73","5,675,047.80","3,925,562.42","25,098,367.24","19,778,266.44","5,320,100.80","6,889,626.64","16,978,907.54","66,963.06","1,420,978.58","3,843,055.98",0.35,0.24
2,Optimista,"16,964,227.07","6,657,184.16","4,907,698.78","25,783,981.75","18,929,321.07","6,854,660.68","7,578,589.31","16,129,962.16","63,614.91","1,420,978.58","4,528,670.49",0.39,0.29


In [11]:
# ==========================================================
# 11. LIMPIEZA Y EXPORTACIÓN
# ==========================================================

def limpiar_exportacion(
    df,
    decimales=2
):
    """
    Limpia infinitos, valores residuales muy pequeños
    y redondea columnas numéricas.
    """

    salida = df.copy()

    salida.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
        inplace=True
    )


    columnas_numericas = (
        salida
        .select_dtypes(
            include=[
                np.number
            ]
        )
        .columns
    )

    for columna in columnas_numericas:

        salida.loc[
            salida[
                columna
            ].abs() < 0.000001,
            columna
        ] = 0.0


        salida[
            columna
        ] = (
            salida[
                columna
            ]
            .round(
                decimales
            )
        )

    return salida


# ==========================================================
# PREPARACIÓN DE EXPORTACIONES
# ==========================================================

resultados_export = (
    resultados_modelos.copy()
)

resultados_export[
    "mae"
] = (
    resultados_export[
        "mae"
    ]
    .round(2)
)

forecast_export = (
    limpiar_exportacion(
        forecast_escenarios,
        decimales=2
    )
)

forecast_powerbi_export = (
    limpiar_exportacion(
        forecast_powerbi,
        decimales=2
    )
)

ratios_export = (
    limpiar_exportacion(
        ratios_forecast,
        decimales=4
    )
)

ratios_powerbi_export = (
    limpiar_exportacion(
        ratios_powerbi,
        decimales=4
    )
)

resumen_export = (
    limpiar_exportacion(
        resumen_escenarios,
        decimales=2
    )
)


# ==========================================================
# EXPORTACIONES
# ==========================================================

EXPORTACIONES = {
    "05_resultados_modelos.csv":
        resultados_export,

    "05_forecast_escenarios.csv":
        forecast_export,

    "05_forecast_powerbi.csv":
        forecast_powerbi_export,

    "05_ratios_forecast.csv":
        ratios_export,

    "05_ratios_forecast_powerbi.csv":
        ratios_powerbi_export,

    "05_resumen_escenarios.csv":
        resumen_export,
}

for nombre, df in EXPORTACIONES.items():

    df.to_csv(
        OUTPUT_DIR / nombre,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )


# ==========================================================
# RESUMEN FINAL
# ==========================================================

print("Archivos exportados:")

for nombre in EXPORTACIONES:

    print(
        f" - data/processed/05/{nombre}"
    )

print(
    f"\nHistórico: "
    f"{historico['periodo'].min():%Y-%m} "
    f"→ "
    f"{historico['periodo'].max():%Y-%m}"
)

print(
    f"Forecast: "
    f"{forecast_base['periodo'].min():%Y-%m} "
    f"→ "
    f"{forecast_base['periodo'].max():%Y-%m}"
)

print(
    f"Meses forecast: "
    f"{forecast_base['periodo'].nunique()}"
)

print(
    "Escenarios: "
    "Pesimista / Base / Optimista"
)

print(
    f"Modelos disponibles: "
    f"{', '.join(MODELOS.keys())}"
)


Archivos exportados:
 - data/processed/05/05_resultados_modelos.csv
 - data/processed/05/05_forecast_escenarios.csv
 - data/processed/05/05_forecast_powerbi.csv
 - data/processed/05/05_ratios_forecast.csv
 - data/processed/05/05_ratios_forecast_powerbi.csv
 - data/processed/05/05_resumen_escenarios.csv

Histórico: 2021-01 → 2023-12
Forecast: 2024-01 → 2024-12
Meses forecast: 12
Escenarios: Pesimista / Base / Optimista
Modelos disponibles: Tendencia, Tendencia + estacionalidad


## Salidas principales

- `05_resultados_modelos.csv`: error de cada modelo y modelo seleccionado.
- `05_forecast_escenarios.csv`: forecast mensual completo.
- `05_forecast_powerbi.csv`: métricas en formato largo.
- `05_ratios_forecast.csv`: ratios previstos por escenario.
- `05_ratios_forecast_powerbi.csv`: ratios previstos en formato largo.
- `05_resumen_escenarios.csv`: resumen anual para la dirección.

### Escenarios utilizados

- **Base:** continuidad de tendencia y estacionalidad.
- **Optimista:** más ventas, mejor control de costes, menor saldo de clientes, más caja y menos deuda.
- **Pesimista:** menos ventas, presión de costes, más clientes pendientes de cobro, menos caja y más deuda.

Los porcentajes de escenario están concentrados en una única función para poder modificarlos fácilmente.